# Старт четвёртого этапа
Пример показывает загрузку и безопасный расчёт одного цифрового отношения на конкретную T. Дальше команда строит собственные снимки, окна и паспорт признаков. Это не готовое решение.


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd

PREPARED = Path("../../lesson2_prepared")  # Измените путь.
NEW_DATA = Path("../data")               # Запуск из lesson4/starter.
TZ = timezone(timedelta(hours=3))
T = datetime(2026, 7, 1, tzinfo=TZ)

def time_moscow(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    return (value.replace(tzinfo=TZ) if value.tzinfo is None else value).astimezone(TZ)

def load(path):
    return pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)

monthly = load(PREPARED / "customer_month.csv")
subscribers = load(PREPARED / "subscribers_clean.csv")
digital = load(NEW_DATA / "digital_usage_monthly.csv")
print(monthly.shape, digital.shape)


In [ ]:
assert not monthly.duplicated(["customer_id", "billing_month"]).any()
assert not digital.duplicated(["customer_id", "usage_month"]).any()
assert set(digital.customer_id) <= set(subscribers.customer_id)
assert set(digital.observation_status) <= {"observed", "not_enabled", "export_gap"}
assert set(zip(digital.customer_id, digital.usage_month)) == set(zip(monthly.customer_id, monthly.billing_month))


In [ ]:
# Для предыдущего месяца: сначала время доступности, затем статус.
previous_month = (T - timedelta(days=1)).replace(day=1).date().isoformat()
known = digital[digital.available_at.map(time_moscow) <= T].copy()
observed = known[(known.usage_month == previous_month) & (known.observation_status == "observed")].copy()
attempts = pd.to_numeric(observed.payment_attempts, errors="raise")
failures = pd.to_numeric(observed.payment_failures, errors="raise")
assert ((failures >= 0) & (failures <= attempts)).all()
observed["digital_failure_rate_last"] = failures / attempts.where(attempts > 0)
print("Известных наблюдаемых отчётов предыдущего месяца:", len(observed))
print("Неизвестная доля при нуле попыток:", observed.digital_failure_rate_last.isna().sum())
# Здесь ещё нет активной базы. При присоединении к ней используйте left join
# и сохраните всех абонентов, даже если их цифровой отчёт отсутствует.


## Продолжение команды
1. Сформируйте активную базу на каждой T по уставу.
2. Рассчитайте признаки одного и трёх месяцев, полноту истории и допустимые отношения.
3. Сохраните X, метки, разбиение и аудит отдельно.
4. Заполните паспорт минимум 12 содержательных признаков.
5. Используйте preprocessing_template.py с собственными списками колонок; fit только на train.
6. Посчитайте train-метрики правила долга и заполните план двух обучаемых кандидатов.
7. Выполните проверки задания, включая добавление будущих строк и новую категорию.
